In [17]:
from pathlib import Path
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = 'iti-unesp-510015'
credential_file = Path.cwd() / 'iti-unesp-510015-1b5281550d01.json'

credentials = service_account.Credentials.from_service_account_file(
    credential_file
)
client = bigquery.Client(project=project_id, credentials=credentials)

query = """
SELECT 
  p.publication_number,
  p.publication_date,
  STRING_AGG(DISTINCT ipc_elem.code, '; ') AS ipc_codes,
  title_localized.text AS title,
  abstract_localized.text AS abstract
FROM 
  `patents-public-data.patents.publications` p,
  UNNEST(p.ipc) AS ipc_elem,
  UNNEST(p.title_localized) AS title_localized,
  UNNEST(p.abstract_localized) AS abstract_localized
WHERE 
  p.country_code = 'BR'
  AND title_localized.language = 'pt'
  AND abstract_localized.language = 'pt'
  AND SUBSTR(ipc_elem.code, 1, 4) IN ('B25J', 'G05B', 'G06F', 'G06N', 'G06Q', 'G05D')
GROUP BY
  p.publication_number,
  p.publication_date,
  title_localized.text,
  abstract_localized.text
"""

# Executa a consulta e gera o DataFrame
df_automacao = client.query(query).to_dataframe()

print(f"Total de patentes de automação encontradas: {len(df_automacao)}")
print(df_automacao.head())

Total de patentes de automação encontradas: 29989
   publication_number  publication_date  \
0  BR-102019027548-A2          20210706   
1  BR-112016015346-A8          20200602   
2     BR-PI0621159-A2          20111129   
3  BR-112016000122-B1          20221101   
4        BR-9714038-A          20000509   

                                           ipc_codes  \
0                               G06F21/62; G06F16/22   
1                               G06Q50/06; G06Q30/02   
2                    G06Q10/00; G06F15/16; G06F17/00   
3  G06F21/62; G06F21/60; G06F21/44; G06F21/31; G0...   
4                                 G05D7/06; G05D7/00   

                                               title  \
0  processo de coleta e agregação de dados numéri...   
1                        sistemas de rede de energia   
2         método implantado por computador e sistema   
3  Método e sistema relacionados à autenticação d...   
4                      Encaixe de regulagem de fluxo   

                 

In [2]:
df_automacao

,publication_number,publication_date,ipc_codes,title,abstract
0,BR-102019027548-A2,20210706,G06F21/62; G06F16/22,processo de coleta e agregação de dados numéri...,processo de coleta e agregação de dados numéri...
1,BR-112016015346-A8,20200602,G06Q50/06; G06Q30/02,sistemas de rede de energia,um sistema inclui um sistema de análise de ser...
2,BR-PI0621159-A2,20111129,G06Q10/00; G06F15/16; G06F17/00,método implantado por computador e sistema,MéTODO IMPLANTADO POR COMPUTADOR E SISTEMA. Tr...
3,BR-112016000122-B1,20221101,G06F21/62; G06F21/60; G06F21/44; G06F21/31; G0...,Método e sistema relacionados à autenticação d...,MÉTODO E SISTEMA RELACIONADOS À AUTENTICAÇÃO D...
4,BR-9714038-A,20000509,G05D7/06; G05D7/00,Encaixe de regulagem de fluxo,&lt;B&gt;&#39;&#39;ENCAIXE DE REGULAGEM DE FLU...
...,...,...,...,...,...
29984,BR-102022006725-A2,20221101,G06Q50/30,Sistema de autenticação,SISTEMA DE AUTENTICAÇÃO. Um sistema de autenti...
29985,BR-102012007224-B1,20201201,G06Q50/30,sistema de controle e gerenciamento de veículo...,sistema de controle e gerenciamento de veículo...
29986,BR-102012007224-B1,20201201,G06Q50/30,sistema de controle e gerenciamento de veículo...,SISTEMA DE CONTROLE E GERENCIAMENTO DE VEÍCULO...
29987,BR-112017027933-A2,20180828,G06Q50/30,método e sistema para determinação de taxa de ...,a presente descrição refere-se a um método e s...


In [3]:
df_automacao.info()

<class 'pandas.DataFrame'>
RangeIndex: 29989 entries, 0 to 29988
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   publication_number  29989 non-null  str  
 1   publication_date    29989 non-null  Int64
 2   ipc_codes           29989 non-null  str  
 3   title               29989 non-null  str  
 4   abstract            29989 non-null  str  
dtypes: Int64(1), str(4)
memory usage: 33.6 MB


In [4]:
df_automacao.to_excel('patentes_automacao_filtradas.xlsx', index=False)

In [5]:
df_automacao_pos2010 = df_automacao[df_automacao['publication_date'] >= 20100101]


In [6]:
df_automacao_pos2010.info()

<class 'pandas.DataFrame'>
Index: 23590 entries, 0 to 29988
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   publication_number  23590 non-null  str  
 1   publication_date    23590 non-null  Int64
 2   ipc_codes           23590 non-null  str  
 3   title               23590 non-null  str  
 4   abstract            23590 non-null  str  
dtypes: Int64(1), str(4)
memory usage: 26.4 MB


In [7]:
df_automacao_limpo = df_automacao_pos2010.drop_duplicates(subset=['publication_number'])

In [8]:
df_automacao_limpo.info()

<class 'pandas.DataFrame'>
Index: 19294 entries, 0 to 29988
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   publication_number  19294 non-null  str  
 1   publication_date    19294 non-null  Int64
 2   ipc_codes           19294 non-null  str  
 3   title               19294 non-null  str  
 4   abstract            19294 non-null  str  
dtypes: Int64(1), str(4)
memory usage: 21.6 MB


In [9]:
df_automacao_limpo.to_excel('df_automacao_limpo.xlsx', index=False)
